In [1]:
# %load_ext ipython_beartype
# %beartype

In [2]:
from pathlib import Path

import rerun as rr
from rerun.catalog import ComponentColumnDescriptor, IndexColumnDescriptor, Schema
from rerun.recording import Recording, load_recording

rrd_path: Path = Path("/mnt/8tb/data/exoego-self-collected/quest+oak+exo/test/session.rrd")
recording: Recording = load_recording(str(rrd_path))


In [3]:
schema: Schema = recording.schema()
# Timelines
timelines: list[IndexColumnDescriptor] = schema.index_columns()
for timeline in timelines:
    print(timeline.name)
    
# Component Columns
components: list[ComponentColumnDescriptor] = schema.component_columns()
# show the first 3 components
for component in components:
    print(component)
    print("---")


log_tick
log_time
video_time
Column name: /:AnnotationContext:context
	Entity path: /
	Archetype: rerun.archetypes.AnnotationContext
	Component type: rerun.components.AnnotationContext
	Component: AnnotationContext:context
	Static: true
---
Column name: /:ViewCoordinates:xyz
	Entity path: /
	Archetype: rerun.archetypes.ViewCoordinates
	Component type: rerun.components.ViewCoordinates
	Component: ViewCoordinates:xyz
	Static: true
---
Column name: /world/ego/left:Transform3D:axis_length
	Entity path: /world/ego/left
	Archetype: rerun.archetypes.Transform3D
	Component type: rerun.components.AxisLength
	Component: Transform3D:axis_length
---
Column name: /world/ego/left:Transform3D:child_frame
	Entity path: /world/ego/left
	Archetype: rerun.archetypes.Transform3D
	Component type: rerun.components.TransformFrameId
	Component: Transform3D:child_frame
---
Column name: /world/ego/left:Transform3D:mat3x3
	Entity path: /world/ego/left
	Archetype: rerun.archetypes.Transform3D
	Component type: rer

## Lets load in all the ego related things

In [4]:
# make sure the timeline exsits
timeline_name = "video_time"
has_timeline: bool = any(timeline.name == timeline_name for timeline in timelines)

# make sure that /world/ego exists
entity_paths: list[str] = sorted({component.entity_path for component in components})
ego_entity_path = Path("/world/ego")
has_ego = any(p == ego_entity_path or p.startswith(f"{ego_entity_path}/") for p in entity_paths)

ego_video_paths:set[Path] = set()
# get all the intrinsics camera components
for component in components:
    if component.archetype == "rerun.archetypes.AssetVideo" and (component.entity_path == ego_entity_path or component.entity_path.startswith(f"{ego_entity_path}/")):
        ego_video_paths.add(Path(component.entity_path))

print(f"Ego video paths: {ego_video_paths}")
cam_names: set[str] = {p.parent.parent.name for p in ego_video_paths}
print(f"Ego camera names: {cam_names}")

Ego video paths: {PosixPath('/world/ego/rgb/pinhole/video'), PosixPath('/world/ego/quest3_right/pinhole/video'), PosixPath('/world/ego/quest3_left/pinhole/video'), PosixPath('/world/ego/right/pinhole/video'), PosixPath('/world/ego/left/pinhole/video')}
Ego camera names: {'right', 'rgb', 'quest3_right', 'quest3_left', 'left'}


In [5]:
import numpy as np
from jaxtyping import Float32, UInt8
from numpy import ndarray
from rerun.dataframe import RecordingView

from simplecv.camera_parameters import Intrinsics

AXIS_CODES: dict[int, str] = {1: "U", 2: "D", 3: "R", 4: "L", 5: "F", 6: "B"}

intri_dict: dict[str, Intrinsics] = {}
for cam_name in cam_names:
    pinhole_entity = ego_entity_path / cam_name / "pinhole"
    assert any(str(pinhole_entity) == c.entity_path for c in components), f"Missing pinhole entity for {cam_name}"

    view: RecordingView = recording.view(index=timeline_name, contents=str(pinhole_entity))
    table = (
        view.filter_index_values(values=[0])
        .select(
            f"{pinhole_entity}:Pinhole:image_from_camera",
            f"{pinhole_entity}:Pinhole:camera_xyz",
            f"{pinhole_entity}:Pinhole:resolution",
        )
        .read_all()
    )

    k_list = table.column(0).to_pylist()[0]
    xyz_list = table.column(1).to_pylist()[0]
    res_list = table.column(2).to_pylist()[0]

    image_from_camera: Float32[ndarray, "3 3"] = np.asarray(k_list, dtype=np.float32).reshape(3, 3, order="F")
    camera_xyz: UInt8[ndarray, "3"] = np.asarray(xyz_list, dtype=np.uint8).reshape(3)
    resolution: Float32[ndarray, "2"] = np.asarray(res_list, dtype=np.float32).reshape(2)

    axes_label: str = "".join(AXIS_CODES[int(v)] for v in camera_xyz)
    assert axes_label in {"RDF"}, f"Unexpected camera axes: {axes_label}"

    intri_dict[cam_name] = Intrinsics.from_k_matrix(
        k_matrix=image_from_camera,
        camera_conventions=axes_label,
        height=int(resolution[1]),
        width=int(resolution[0]),
    )

print(intri_dict)

{'right': Intrinsics(camera_conventions=RDF, fl_x=571.6647338867188, fl_y=571.4653930664062, cx=650.5889282226562, cy=350.9523010253906, height=720, width=1280), 'rgb': Intrinsics(camera_conventions=RDF, fl_x=765.0453491210938, fl_y=764.6630249023438, cx=635.09228515625, cy=356.91204833984375, height=720, width=1280), 'quest3_right': Intrinsics(camera_conventions=RDF, fl_x=853.807861328125, fl_y=853.807861328125, cx=636.6122436523438, cy=481.6759338378906, height=960, width=1280), 'quest3_left': Intrinsics(camera_conventions=RDF, fl_x=859.751708984375, fl_y=859.751708984375, cx=642.2196655273438, cy=482.0271301269531, height=960, width=1280), 'left': Intrinsics(camera_conventions=RDF, fl_x=572.2032470703125, fl_y=572.0864868164062, cx=648.4393310546875, cy=356.680419921875, height=720, width=1280)}


In [10]:
import numpy as np
from jaxtyping import Float32


def load_extrinsics_series(recording: Recording, entity: str, timeline: str) -> tuple[
     Float32[ndarray, "n 3 3"], Float32[ndarray, "n 3"]
]:
    view: RecordingView = recording.view(index=timeline, contents=entity)
    tbl = view.select(
        timeline,
        f"{entity}:Transform3D:mat3x3",
        f"{entity}:Transform3D:translation",
    ).read_all()


    def unwrap(v):
        return v[0] if isinstance(v, list) and len(v) == 1 else v

    mats = [
        np.asarray(unwrap(m), np.float32).reshape(3, 3, order="F")
        for m in tbl.column(1).to_pylist()
    ]
    trans = [
        np.asarray(unwrap(t), np.float32).reshape(-1)[:3]
        for t in tbl.column(2).to_pylist()
    ]

    cam_R_world_batch: Float32[ndarray, "n 3 3"] = np.stack(mats, axis=0)
    cam_t_world_batch: Float32[ndarray, "n 3"] = np.stack(trans, axis=0)
    return cam_R_world_batch, cam_t_world_batch
# 2. load the extrinsics
for cam_name in cam_names:
    transform_entity: Path = ego_entity_path / cam_name
    assert any(str(transform_entity) == component.entity_path for component in components), f"Missing pinhole entity for camera {cam_name}"

    cam_R_world_batch, cam_t_world_batch = load_extrinsics_series(
        recording=recording,
        entity=str(transform_entity),
        timeline=timeline_name,
    )
    
